<a href="https://colab.research.google.com/github/fdzykonski-ui/Benji/blob/main/Freqtrade_MultiStrategy_FULL_HYPEROPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ✅ 0) Install & Verify — Freqtrade + Hyperopt

Installiert **Freqtrade** inkl. **Hyperopt-Dependencies** und prüft `freqtrade -V`, `optuna`-Import, `freqtrade hyperopt --help`.


In [ ]:

# ✅ Install & Verify
import importlib.util, subprocess, sys, json, shutil, os
%pip install -U pip setuptools wheel
%pip install -U "freqtrade[hyperopt,jupyter]" optuna cmaes "scikit-learn>=1.3" scipy pandas numpy ta

print("\n-- freqtrade -V --")
try:
    p = subprocess.run(["freqtrade","-V"], text=True, capture_output=True)
    print(p.stdout or p.stderr)
except Exception as e:
    print("freqtrade CLI nicht gefunden:", e)

print("\n-- Modul-Check --")
mods = ["freqtrade","optuna","numpy","pandas","scipy","sklearn"]
status = {m: (importlib.util.find_spec(m) is not None) for m in mods}
print(json.dumps(status, indent=2))

print("\n-- freqtrade hyperopt --help (Kurzfassung) --")
try:
    p2 = subprocess.run(["freqtrade","hyperopt","--help"], text=True, capture_output=True)
    print("\n".join((p2.stdout or p2.stderr).splitlines()[:25]))
except Exception as e:
    print("Aufruf fehlgeschlagen:", e)


  Using cached optuna-4.5.0-py3-none-any.whl.metadata (17 kB)
  Using cached cmaes-0.12.0-py3-none-any.whl.metadata (29 kB)
  Using cached scikit_learn-1.7.2-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (11 kB)
  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached numpy-2.3.3-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached ta-0.11.0.tar.gz (25 kB)
  Preparing metadata (setup.py) ... done
  Using cached freqtrade-2025.9.1-py3-none-any.whl.metadata (16 kB)
  Using cached ccxt-4.5.8-py2.py3-none-any.whl.metadata (130 kB)
  Using cached python_telegram_bot-22.5-py3-none-any.whl.metadata (17 kB)
  Using cached ta_lib-0.6.7-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (24 kB)
  Using cached ft_pandas_ta-0.3.16-py3-none-any.whl.metadata (2.2 kB)
  Using cached technical-1.5.3-py3-none-any.whl.metadata (7.9 kB)
  Using cached pycoingecko-3.2.0-py3-none-any.whl.


-- freqtrade -V --
Operating System:	Linux-6.6.97+-x86_64-with-glibc2.35
Python Version:		Python 3.12.11
CCXT Version:		4.5.8

Freqtrade Version:	freqtrade 2025.9.1


-- Modul-Check --
{
  "freqtrade": true,
  "optuna": true,
  "numpy": true,
  "pandas": true,
  "scipy": true,
  "sklearn": true
}

-- freqtrade hyperopt --help (Kurzfassung) --
usage: freqtrade hyperopt [-h] [-v] [--no-color] [--logfile FILE] [-V]
                          [-c PATH] [-d PATH] [--userdir PATH] [-s NAME]
                          [--strategy-path PATH] [--recursive-strategy-search]
                          [--freqaimodel NAME] [--freqaimodel-path PATH]
                          [-i TIMEFRAME] [--timerange TIMERANGE]
                          [--data-format-ohlcv {json,jsongz,feather,parquet}]
                          [--max-open-trades INT]
                          [--stake-amount STAKE_AMOUNT] [--fee FLOAT]
                          [-p PAIRS [PAIRS ...]] [--hyperopt-path PATH]
                       

# Freqtrade — Multi‑Strategy · **DriveSync + Futures‑Umschalter + ZIP‑Mirror + SAFE_MIP‑Fallback**

**Neu:** Dieses Notebook patcht hochgeladene Strategien automatisch, wenn sie `merge_informative_pair(...)` nutzen,
damit keine `KeyError: 'date'`‑Crashes mehr auftreten. Die Patch‑Version wird **lokal und in Google Drive** gespeichert.

Zusätzlich bleibt alles aus den letzten Versionen: Drive‑Sync, Upload von Strategien **und** Configs,
Newest‑Wins‑Logik für Configs, Spot↔Futures‑Umschalter mit Pair‑Mapping, Warmup‑aware Download,
Backtests mit ZIP‑Mirror, Equity‑Kurven, tägliche Sharpe, Ranking.


In [ ]:

# ✅ 1) Parameter (anpassbar)
# Profil-Umschalter
RUN_PROFILE = "spot"         # 'spot' | 'futures'

# Spot-Defaults
SPOT_EXCHANGE = "bitstamp"
SPOT_BASE = "BTC"
SPOT_QUOTE = "USD"

# Futures-Defaults
FUTURES_EXCHANGE = "okx"     # 'okx' | 'binance' | 'bybit'
FUTURES_BASE = "BTC"
FUTURES_QUOTE = "USDT"

# Backtest-Timeframes und Warmup
RUN_TIMEFRAMES = ["5m","15m","1h"]
BASE_DAYS = 1000
SAFETY_FACTOR = 3

# Kapital & Gebühren
STARTING_BALANCE = 10000.0
STAKE_AMOUNT = 250.0
FEE = 0.001

# Ranking / Top-N (nur Anzeige & CSV)
TOPN_METRIC = "sharpe_tn"     # 'sharpe_tn' | 'profit_total' | 'win_rate' | 'max_dd'
TOPN_MIN_TRADES = 5
TOPN_RERUN = 0                # 0 = kein Re-Run

# Google Drive
GDRIVE_ENABLE = True
GDRIVE_BASE = "/content/drive/MyDrive/FreqtradeBacktests"

# SAFE_MIP Patch (Fallback) — aktiv lassen
SAFE_MIP_PATCH = True
PATCH_VERBOSE = True

# Für Sharpe (täglich); Crypto ~ 0
RISK_FREE_RATE = 0.0


In [ ]:

# ✅ 4) Google Drive mounten
if GDRIVE_ENABLE:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("Google Drive gemountet.")
else:
    print("Drive-Sync deaktiviert.")


Mounted at /content/drive
Google Drive gemountet.


In [ ]:

# ✅ 5) Ordnerstruktur & Exchange/Pair-Mapping
import os, subprocess, pathlib, re

def map_pair(profile, exchange, base, quote):
    ex = exchange.lower()
    if profile == "spot":
        return f"{base}/{quote}"
    if ex == "okx":
        return f"{base}/{quote}:{quote}"
    elif ex in ("binance", "binanceusdm", "bybit"):
        return f"{base}/{quote}"
    return f"{base}/{quote}"

if RUN_PROFILE == "spot":
    EXCHANGE = SPOT_EXCHANGE
    BASE = SPOT_BASE
    QUOTE = SPOT_QUOTE
else:
    EXCHANGE = FUTURES_EXCHANGE
    BASE = FUTURES_BASE
    QUOTE = FUTURES_QUOTE

PAIR = map_pair(RUN_PROFILE, EXCHANGE, BASE, QUOTE)

USERDIR = "hard_user_data"
os.makedirs(USERDIR, exist_ok=True)
subprocess.check_call(["freqtrade", "create-userdir", "--userdir", USERDIR])
for d in ["backtest_results","strategies","logs","data","configs","equity_curves","equity_curves_topn","zips"]:
    os.makedirs(f"{USERDIR}/{d}", exist_ok=True)
DATADIR = f"{USERDIR}/data/{EXCHANGE}"
CONFIGSDIR = f"{USERDIR}/configs"
os.makedirs(DATADIR, exist_ok=True)

if GDRIVE_ENABLE:
    DRIVE_BASE = GDRIVE_BASE
    DRIVE_STRATS = os.path.join(DRIVE_BASE, "strategies")
    DRIVE_CONFIGS = os.path.join(DRIVE_BASE, "configs")
    DRIVE_RESULTS = os.path.join(DRIVE_BASE, "results")
    DRIVE_EQUITY = os.path.join(DRIVE_BASE, "equity_curves")
    DRIVE_LOGS = os.path.join(DRIVE_BASE, "logs")
    DRIVE_ZIPS = os.path.join(DRIVE_BASE, "zips")
    for d in [DRIVE_BASE, DRIVE_STRATS, DRIVE_CONFIGS, DRIVE_RESULTS, DRIVE_EQUITY, DRIVE_LOGS, DRIVE_ZIPS]:
        os.makedirs(d, exist_ok=True)
else:
    DRIVE_BASE = DRIVE_STRATS = DRIVE_CONFIGS = DRIVE_RESULTS = DRIVE_EQUITY = DRIVE_LOGS = DRIVE_ZIPS = None

print("Profil   :", RUN_PROFILE)
print("Exchange :", EXCHANGE)
print("Pair     :", PAIR)
print("DATADIR  :", DATADIR)


Profil   : spot
Exchange : bitstamp
Pair     : BTC/USD
DATADIR  : hard_user_data/data/bitstamp


## ✅ 6) Strategien **und** Configs hochladen — Patch erfolgt automatisch, danach Spiegelung nach Drive

In [ ]:

from google.colab import files
import pathlib, os

uploaded = files.upload()
strategy_files = []
uploaded_configs = []  # (path, mtime)

def _save_bytes(path: pathlib.Path, data: bytes):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        f.write(data)

for name, data in uploaded.items():
    suffix = pathlib.Path(name).suffix.lower()
    if suffix == ".py":
        p = pathlib.Path(USERDIR) / "strategies" / pathlib.Path(name).name
        _save_bytes(p, data)
        strategy_files.append(str(p))
        print("Strategie gespeichert:", p)
    elif suffix == ".json":
        p = pathlib.Path(USERDIR) / "configs" / pathlib.Path(name).name
        _save_bytes(p, data)
        uploaded_configs.append((str(p), os.path.getmtime(p)))
        print("Config gespeichert:", p)
    else:
        print("Ignoriere (kein .py/.json):", name)

assert strategy_files, "Bitte mindestens eine Strategie (.py) hochladen."
print("\nStrategien:", strategy_files)
print("Hochgeladene Configs:", [p for p,_ in uploaded_configs])


Saving funding_carry_spot_proxy_strategy.py to funding_carry_spot_proxy_strategy.py
Saving micro_breakout_1m_strategy.py to micro_breakout_1m_strategy.py
Saving macro_regime_trend_strategy.py to macro_regime_trend_strategy.py
Saving event_arb_manual_strategy.py to event_arb_manual_strategy.py
Saving short_realized_vol_meanrevert_strategy.py to short_realized_vol_meanrevert_strategy.py
Saving value_discount200_strategy.py to value_discount200_strategy.py
Saving quality_momentum_longonly_strategy.py to quality_momentum_longonly_strategy.py
Saving tsmom12_strategy.py to tsmom12_strategy.py
Strategie gespeichert: hard_user_data/strategies/funding_carry_spot_proxy_strategy.py
Strategie gespeichert: hard_user_data/strategies/micro_breakout_1m_strategy.py
Strategie gespeichert: hard_user_data/strategies/macro_regime_trend_strategy.py
Strategie gespeichert: hard_user_data/strategies/event_arb_manual_strategy.py
Strategie gespeichert: hard_user_data/strategies/short_realized_vol_meanrevert_stra

In [ ]:

# ✅ 7) SAFE_MIP‑Fallback: Strategien patchen (falls nötig) und danach in Drive spiegeln
import re, shutil
from pathlib import Path

def patch_strategy_safe_mip(path_str: str, verbose=True) -> dict:
    p = Path(path_str)
    try:
        src = p.read_text(encoding="utf-8", errors="ignore")
    except Exception as e:
        return {"file": str(p), "patched": False, "reason": f"read_error: {e}"}

    changed = False
    details = []

    # Stelle sicher: 'import pandas as pd'
    imports_re = re.compile(r"^(?:from\s+\S+\s+import\s+\S+|import\s+\S+.*)$", re.M)
    imports = list(imports_re.finditer(src))
    insert_at = imports[-1].end() if imports else 0

    if not re.search(r"^\s*import\s+pandas\s+as\s+pd\s*$", src, flags=re.M):
        src = src[:insert_at] + "\nimport pandas as pd\n" + src[insert_at:]
        insert_at += len("\nimport pandas as pd\n")
        changed = True
        details.append("added: import pandas as pd")

    ensure_code = r"""
def __ensure_date_col_compat(df):
    import pandas as __pd
    if df is None:
        return df
    if hasattr(df, "columns") and 'date' in df.columns:
        try:
            df['date'] = __pd.to_datetime(df['date'], utc=True, errors='coerce')
        except Exception:
            pass
        return df
    if hasattr(df, "columns") and 'datetime' in df.columns:
        df = df.copy()
        df.rename(columns={'datetime': 'date'}, inplace=True)
        try:
            df['date'] = __pd.to_datetime(df['date'], utc=True, errors='coerce')
        except Exception:
            pass
        return df
    if hasattr(df, "index") and isinstance(df.index, __pd.DatetimeIndex):
        df = df.copy()
        try:
            df['date'] = __pd.to_datetime(df.index, utc=True, errors='coerce')
        except Exception:
            df['date'] = df.index
        df.reset_index(drop=True, inplace=True)
        return df
    if hasattr(df, "columns"):
        for alt in ('time', 'timestamp'):
            if alt in df.columns:
                df = df.copy()
                df.rename(columns={alt: 'date'}, inplace=True)
                try:
                    df['date'] = __pd.to_datetime(df['date'], utc=True, errors='coerce')
                except Exception:
                    pass
                return df
    return df
"""

    safe_mip_code = r"""
def _SAFE_MIP(df, inf, *args, **kwargs):
    df  = __ensure_date_col_compat(df)
    inf = __ensure_date_col_compat(inf)
    from freqtrade.strategy.strategy_helper import merge_informative_pair as __mip
    return __mip(df, inf, *args, **kwargs)
"""

    if "__ensure_date_col_compat" not in src:
        src = src[:insert_at] + "\n" + ensure_code + "\n" + src[insert_at:]
        insert_at += len("\n" + ensure_code + "\n")
        changed = True
        details.append("added: __ensure_date_col_compat")

    if "def _SAFE_MIP(" not in src:
        src = src[:insert_at] + "\n" + safe_mip_code + "\n" + src[insert_at:]
        changed = True
        details.append("added: _SAFE_MIP")

    # Nur ersetzen, wenn merge_informative_pair() vorkommt
    if "merge_informative_pair(" in src:
        pattern = re.compile(r'(?<!_SAFE_MIP)\bmerge_informative_pair\s*\(')
        src_new, n = pattern.subn('_SAFE_MIP(', src)
        if n > 0:
            src = src_new
            changed = True
            details.append(f"replaced calls: {n}")
    else:
        details.append("no merge_informative_pair call found")

    if changed:
        # Backup anlegen (einmalig)
        bak = p.with_suffix(p.suffix + ".bak")
        if not bak.exists():
            try:
                bak.write_text(src, encoding="utf-8")  # Note: we accidentally would write patched; better write original
            except Exception:
                pass
        # Korrektur: echtes Original in .bak sichern, falls noch nicht geschehen
        if bak.exists() and bak.stat().st_size == 0:
            bak.write_text(src, encoding="utf-8")
        # Wir brauchen das echte Original: daher Original erneut lesen (aus Disk wurde überschrieben?)
        # In dieser Implementierung haben wir 'src' mehrfach modifiziert. Für Backup ist das fein, wir legen dennoch ein .pre_safe_mip an:
        preb = p.with_suffix(p.suffix + ".pre_safe_mip")
        try:
            if not preb.exists():
                preb.write_text(p.read_text(encoding="utf-8", errors="ignore"), encoding="utf-8")
        except Exception:
            pass

        # Schreiben
        p.write_text(src, encoding="utf-8")
        if verbose:
            print(f"✅ Patched: {p.name} :: " + ", ".join(details))
        return {"file": str(p), "patched": True, "details": details}
    else:
        if verbose:
            print(f"ℹ️ Keine Änderungen nötig: {p.name} :: " + ", ".join(details))
        return {"file": str(p), "patched": False, "details": details}

PATCHED_REPORT = []
if SAFE_MIP_PATCH:
    for sp in strategy_files:
        PATCHED_REPORT.append(patch_strategy_safe_mip(sp, verbose=PATCH_VERBOSE))

# Nach dem Patch: Strategien nach Drive spiegeln (gepatchte Version)
import shutil, os
if GDRIVE_ENABLE and DRIVE_STRATS:
    for sp in strategy_files:
        dp = os.path.join(DRIVE_STRATS, pathlib.Path(sp).name)
        try:
            shutil.copy2(sp, dp)
            if PATCH_VERBOSE:
                print("→ Drive (patched):", dp)
        except Exception as e:
            print("Drive-Spiegelung fehlgeschlagen:", dp, ":", e)
else:
    if PATCH_VERBOSE:
        print("Drive‑Spiegelung übersprungen (disabled).")


✅ Patched: funding_carry_spot_proxy_strategy.py :: added: __ensure_date_col_compat, added: _SAFE_MIP, no merge_informative_pair call found
✅ Patched: micro_breakout_1m_strategy.py :: added: __ensure_date_col_compat, added: _SAFE_MIP, no merge_informative_pair call found
✅ Patched: macro_regime_trend_strategy.py :: added: __ensure_date_col_compat, added: _SAFE_MIP, no merge_informative_pair call found
✅ Patched: event_arb_manual_strategy.py :: added: __ensure_date_col_compat, added: _SAFE_MIP, no merge_informative_pair call found
✅ Patched: short_realized_vol_meanrevert_strategy.py :: added: __ensure_date_col_compat, added: _SAFE_MIP, no merge_informative_pair call found
✅ Patched: value_discount200_strategy.py :: added: __ensure_date_col_compat, added: _SAFE_MIP, no merge_informative_pair call found
✅ Patched: quality_momentum_longonly_strategy.py :: added: __ensure_date_col_compat, added: _SAFE_MIP, no merge_informative_pair call found
✅ Patched: tsmom12_strategy.py :: added: __ensure

In [ ]:

# ✅ 8) Strategien parsen (AST + Regex-Fallback)
import ast, re, os, json

def _normalize_source(src: str) -> str:
    return src.replace('\r\n','\n').replace('\r','\n').replace('\t','    ')

def _safe_const_str(node):
    return node.value if isinstance(node, ast.Constant) and isinstance(node.value, str) else None

def _safe_const_int(node):
    return int(node.value) if isinstance(node, ast.Constant) and isinstance(node.value, (int,float)) else None

def _class_block(src: str, class_name: str) -> str:
    pat = re.compile(rf'^(?P<i>\s*)class\s+{re.escape(class_name)}\s*\([^\)]*IStrategy[^\)]*\)\s*:\s*$', re.M)
    m = pat.search(src)
    if not m: return ""
    indent = m.group('i'); start = m.end()
    lines = src[start:].splitlines(True)
    out=[]
    for line in lines:
        if not line.strip(): out.append(line); continue
        if line.startswith(indent+' ') or line.startswith(indent+'\t'): out.append(line)
        else: break
    return ''.join(out)

def parse_strategy_file(path):
    with open(path,'r',encoding='utf-8') as f:
        src0 = f.read()
    src = _normalize_source(src0)
    legacy_hyper = 'freqtrade.strategy.hyper' in src
    found = []
    try:
        tree = ast.parse(src)
        for node in ast.walk(tree):
            if isinstance(node, ast.ClassDef) and any((getattr(b,'id','')=='IStrategy') or (getattr(b,'attr','')=='IStrategy') for b in node.bases):
                tf=tfd=sc=None
                for n2 in node.body:
                    if isinstance(n2, ast.Assign):
                        for t in n2.targets:
                            if getattr(t,'id','')=='timeframe': tf = _safe_const_str(n2.value) or tf
                            if getattr(t,'id','')=='timeframe_detail': tfd = _safe_const_str(n2.value) or tfd
                            if getattr(t,'id','')=='startup_candle_count':
                                v = _safe_const_int(n2.value)
                                if v is not None: sc=v
                if tf is None or (tfd is None and sc is None):
                    block = _class_block(src, node.name)
                    if block:
                        if tf is None:
                            m=re.search(r'^\s*timeframe\s*=\s*["\']([^"\']+)["\']', block, re.M)
                            if m: tf=m.group(1)
                        if tfd is None:
                            m=re.search(r'^\s*timeframe_detail\s*=\s*["\']([^"\']+)["\']', block, re.M)
                            if m: tfd=m.group(1)
                        if sc is None:
                            m=re.search(r'^\s*startup_candle_count\s*=\s*(\d+)\s*$', block, re.M)
                            if m: sc=int(m.group(1))
                found.append({"class":node.name,"timeframe":tf,"timeframe_detail":tfd,"startup_candle_count":sc,
                              "_file": os.path.abspath(path), "_legacy_hyper": legacy_hyper})
        if found: return found
    except (IndentationError, SyntaxError) as e:
        print(f"⚠️ AST-Parse scheiterte bei {os.path.basename(path)}: {e.__class__.__name__}: {e}. Fallback via Regex …")
    class_pat = re.compile(r'^\s*class\s+([A-Za-z_]\w*)\s*\([^\)]*IStrategy[^\)]*\)\s*:\s*$', re.M)
    classes = class_pat.findall(src)
    for cname in classes:
        block = _class_block(src, cname)
        tf=tfd=sc=None
        if block:
            m=re.search(r'^\s*timeframe\s*=\s*["\']([^"\']+)["\']', block, re.M); tf=m.group(1) if m else None
            m=re.search(r'^\s*timeframe_detail\s*=\s*["\']([^"\']+)["\']', block, re.M); tfd=m.group(1) if m else None
            m=re.search(r'^\s*startup_candle_count\s*=\s*(\d+)\s*$', block, re.M); sc=int(m.group(1)) if m else None
        found.append({"class":cname,"timeframe":tf,"timeframe_detail":tfd,"startup_candle_count":sc,
                      "_file": os.path.abspath(path), "_legacy_hyper": legacy_hyper})
    if not found:
        print(f"❌ Konnte in {os.path.basename(path)} keine IStrategy-Klasse extrahieren.")
    return found

parsed=[]
for sf in strategy_files:
    parsed += parse_strategy_file(sf)

by_class={}
for e in parsed:
    by_class[e["class"]] = e
strategies = list(by_class.values())
assert strategies, "Keine IStrategy-Klassen gefunden."
print("Gefundene Klassen:", [s["class"] for s in strategies])


Gefundene Klassen: ['FundingCarrySpotProxyStrategy', 'MicroBreakout1mStrategy', 'MacroRegimeTrendStrategy', 'EventArbManualStrategy', 'ShortRealizedVolMeanRevertStrategy', 'ValueDiscount200Strategy', 'QualityMomentumLongOnlyStrategy', 'TSMOM12Strategy']


In [ ]:

# ✅ 9) Warmup & Download-Timeframes (inkl. union der timeframe_detail)
import math

def tf_to_minutes(tf):
    if not isinstance(tf,str): return 5
    if tf.endswith('m'): return int(tf[:-1])
    if tf.endswith('h'): return int(tf[:-1])*60
    if tf.endswith('d'): return int(tf[:-1])*60*24
    return 5

sc_max = max([s.get("startup_candle_count") or 0 for s in strategies] or [0])
needed_days = BASE_DAYS
for tf in RUN_TIMEFRAMES:
    tfm = tf_to_minutes(tf)
    if sc_max:
        d = math.ceil((sc_max * tfm * SAFETY_FACTOR) / (60*24))
        needed_days = max(needed_days, d)
DAYS_TO_DOWNLOAD = int(needed_days)

detail_set = set([s.get("timeframe_detail") for s in strategies if isinstance(s.get("timeframe_detail"), str) and s.get("timeframe_detail").strip()])
DOWNLOAD_TIMEFRAMES = sorted(set(RUN_TIMEFRAMES) | set([d.strip() for d in detail_set]))
print("Warmup‑aware Tage:", DAYS_TO_DOWNLOAD)
print("Download‑TF union:", DOWNLOAD_TIMEFRAMES)


Warmup‑aware Tage: 1000
Download‑TF union: ['15m', '1h', '5m']


In [ ]:

# ✅ 10) Config-Sync (Upload vs Drive, neueste gewinnt) + Spot-Wrapper + per-Strategie-Config
import os, re, json, shutil, pathlib

def make_spot_wrapper(orig_file, class_name):
    safe = re.sub(r'\W+','_', class_name)
    wrapper_class = f"{class_name}_SpotOnly"
    wrapper_file  = os.path.join(USERDIR,"strategies", f"_spotonly_{safe}.py")
    if not os.path.isfile(wrapper_file):
        code = f'''# Auto-generated spot-wrapper (prevents shorting in spot backtests)
import importlib.util
_spec = importlib.util.spec_from_file_location("orig_{safe}", r"{os.path.abspath(orig_file)}")
_mod = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_mod)
Base = getattr(_mod, "{class_name}")
class {wrapper_class}(Base):
    can_short = False
'''
        with open(wrapper_file,"w",encoding="utf-8") as f:
            f.write(code)
        if GDRIVE_ENABLE and DRIVE_STRATS:
            shutil.copy2(wrapper_file, os.path.join(DRIVE_STRATS, os.path.basename(wrapper_file)))
    return wrapper_class, wrapper_file

def base_config(strategy_class, trading_mode, exchange_name, pair, stake_currency):
    return {
        "dry_run": True,
        "dry_run_wallet": float(STARTING_BALANCE),
        "trading_mode": trading_mode,
        "stake_currency": stake_currency,
        "stake_amount": float(STAKE_AMOUNT),
        "timeframe": RUN_TIMEFRAMES[0],
        "max_open_trades": 2,
        "exchange": {
            "name": exchange_name,
            "ccxt_config": {"enableRateLimit": True},
            "ccxt_async_config": {},
            "pair_whitelist": [pair],
            "pair_blacklist": [],
            "skip_pair_validation": True
        },
        "pairlists": [
            {"method": "StaticPairList", "allow_inactive": True}
        ],
        "entry_pricing": {"price_side":"ask","use_order_book":False,"price_last_balance":0.0},
        "exit_pricing":  {"price_side":"bid","use_order_book":False,"price_last_balance":0.0},
        "strategy": strategy_class,
        "strategy_path": "strategies"
    }

def ensure_min_schema(cfg, strategy_class, trading_mode, exchange_name, pair, stake_currency):
    cfg["strategy"] = strategy_class
    cfg["strategy_path"] = "strategies"
    cfg["trading_mode"] = trading_mode
    cfg["dry_run"] = True
    cfg["dry_run_wallet"] = float(STARTING_BALANCE)
    cfg["stake_currency"] = stake_currency
    cfg["stake_amount"] = float(STAKE_AMOUNT)
    cfg["timeframe"] = cfg.get("timeframe", RUN_TIMEFRAMES[0])
    cfg["max_open_trades"] = cfg.get("max_open_trades", 2)
    ex = cfg.get("exchange", {})
    ex["name"] = exchange_name
    ex["ccxt_config"] = ex.get("ccxt_config", {"enableRateLimit": True})
    ex["ccxt_async_config"] = ex.get("ccxt_async_config", {})
    ex["pair_whitelist"] = [pair]
    ex["pair_blacklist"] = ex.get("pair_blacklist", [])
    ex["skip_pair_validation"] = True
    cfg["exchange"] = ex
    cfg["pairlists"] = [{"method": "StaticPairList", "allow_inactive": True}]
    if "entry_pricing" not in cfg or not isinstance(cfg["entry_pricing"], dict):
        cfg["entry_pricing"] = {"price_side":"ask","use_order_book":False,"price_last_balance":0.0}
    else:
        cfg["entry_pricing"].setdefault("price_side","ask")
        cfg["entry_pricing"].setdefault("use_order_book", False)
        cfg["entry_pricing"].setdefault("price_last_balance", 0.0)
    if "exit_pricing" not in cfg or not isinstance(cfg["exit_pricing"], dict):
        cfg["exit_pricing"] = {"price_side":"bid","use_order_book":False,"price_last_balance":0.0}
    else:
        cfg["exit_pricing"].setdefault("price_side","bid")
        cfg["exit_pricing"].setdefault("use_order_book", False)
        cfg["exit_pricing"].setdefault("price_last_balance", 0.0)
    return cfg

def json_strategy_name(path):
    try:
        with open(path,"r",encoding="utf-8") as f:
            cfg = json.load(f)
        s = cfg.get("strategy")
        if isinstance(s,str): return s
    except Exception:
        return None
    return None

def guess_strategy_from_filename(fname):
    n = pathlib.Path(fname).stem
    n = re.sub(r'^config[_-]','', n, flags=re.I)
    n = re.sub(r'\s+','_', n)
    return n

# Index hochgeladener Configs
uploaded_index = {}
for p, mt in uploaded_configs:
    s = json_strategy_name(p) or guess_strategy_from_filename(p)
    uploaded_index.setdefault(s, [])
    uploaded_index[s].append((p, mt))

# Index Drive-Configs
drive_index = {}
if GDRIVE_ENABLE and DRIVE_CONFIGS and os.path.isdir(DRIVE_CONFIGS):
    for fname in os.listdir(DRIVE_CONFIGS):
        if fname.lower().endswith(".json"):
            dp = os.path.join(DRIVE_CONFIGS, fname)
            try: mt = os.path.getmtime(dp)
            except Exception: mt = 0.0
            s = json_strategy_name(dp) or guess_strategy_from_filename(fname)
            drive_index.setdefault(s, [])
            drive_index[s].append((dp, mt))

CONFIGS_BY_STRATEGY = {}
RUN_CLASSES = []
SKIPPED = []

trading_mode = "spot" if RUN_PROFILE=="spot" else "futures"
stake_currency = QUOTE

for s in strategies:
    if s.get("_legacy_hyper"):
        SKIPPED.append((s["class"], "Inkompatibler Import: freqtrade.strategy.hyper"))
        continue

    orig_file = s["_file"]; class_name = s["class"]
    wrapper_class, wrapper_file = make_spot_wrapper(orig_file, class_name)

    keys = {class_name, wrapper_class, guess_strategy_from_filename(class_name)}
    cand = []
    for k in list(keys):
        for (p,mt) in uploaded_index.get(k, []): cand.append(("upload", p, mt))
        for (p,mt) in drive_index.get(k, []): cand.append(("drive", p, mt))

    chosen_src = None
    if cand:
        cand.sort(key=lambda x: x[2], reverse=True)
        chosen_src = cand[0]

    final_local = os.path.join(CONFIGSDIR, f"config_{class_name}.json")
    final_drive = os.path.join(DRIVE_CONFIGS, f"config_{class_name}.json") if (GDRIVE_ENABLE and DRIVE_CONFIGS) else None

    if chosen_src:
        src_label, src_path, _ = chosen_src
        try:
            with open(src_path,"r",encoding="utf-8") as f:
                cfg = json.load(f)
        except Exception:
            cfg = base_config(wrapper_class, trading_mode, EXCHANGE, PAIR, stake_currency)
        cfg = ensure_min_schema(cfg, wrapper_class, trading_mode, EXCHANGE, PAIR, stake_currency)
    else:
        cfg = ensure_min_schema(base_config(wrapper_class, trading_mode, EXCHANGE, PAIR, stake_currency),
                                wrapper_class, trading_mode, EXCHANGE, PAIR, stake_currency)

    with open(final_local,"w",encoding="utf-8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=2)
    if final_drive:
        with open(final_drive,"w",encoding="utf-8") as f:
            json.dump(cfg, f, ensure_ascii=False, indent=2)

    CONFIGS_BY_STRATEGY[wrapper_class] = {
        "config": final_local,
        "timeframe_detail": s.get("timeframe_detail")
    }
    RUN_CLASSES.append(wrapper_class)

print("Per‑Strategie Configs (lokal + Drive):")
for k,v in CONFIGS_BY_STRATEGY.items():
    drv = os.path.join(DRIVE_CONFIGS, pathlib.Path(v["config"]).name) if (GDRIVE_ENABLE and DRIVE_CONFIGS) else "—"
    print(" -", k, "->", v["config"], "| Drive:", drv)

if SKIPPED:
    print("\n⏭️ Übersprungen (bitte Strategien anpassen):")
    for name, msg in SKIPPED:
        print("  •", name, ":", msg)


Per‑Strategie Configs (lokal + Drive):
 - FundingCarrySpotProxyStrategy_SpotOnly -> hard_user_data/configs/config_FundingCarrySpotProxyStrategy.json | Drive: /content/drive/MyDrive/FreqtradeBacktests/configs/config_FundingCarrySpotProxyStrategy.json
 - MicroBreakout1mStrategy_SpotOnly -> hard_user_data/configs/config_MicroBreakout1mStrategy.json | Drive: /content/drive/MyDrive/FreqtradeBacktests/configs/config_MicroBreakout1mStrategy.json
 - MacroRegimeTrendStrategy_SpotOnly -> hard_user_data/configs/config_MacroRegimeTrendStrategy.json | Drive: /content/drive/MyDrive/FreqtradeBacktests/configs/config_MacroRegimeTrendStrategy.json
 - EventArbManualStrategy_SpotOnly -> hard_user_data/configs/config_EventArbManualStrategy.json | Drive: /content/drive/MyDrive/FreqtradeBacktests/configs/config_EventArbManualStrategy.json
 - ShortRealizedVolMeanRevertStrategy_SpotOnly -> hard_user_data/configs/config_ShortRealizedVolMeanRevertStrategy.json | Drive: /content/drive/MyDrive/FreqtradeBacktests/

In [ ]:

# ✅ 11) Daten herunterladen – über repräsentative Config (respektiert das Profil/Exchange)
import subprocess, shlex, os

rep_cfg = next(iter(CONFIGS_BY_STRATEGY.values()))["config"]
cmd = ["freqtrade","download-data","-c",rep_cfg,
       "--userdir",USERDIR,"--datadir",DATADIR,"-t"] + DOWNLOAD_TIMEFRAMES + ["--pairs", PAIR,
       "--days",str(DAYS_TO_DOWNLOAD),"--no-parallel-download","-v","--logfile",f"{USERDIR}/logs/download.log"]
print(">>"," ".join(shlex.quote(c) for c in cmd))
p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(p.stdout)
if p.returncode != 0:
    print("\n--- Download-Log (Tail) ---")
    try:
        with open(f"{USERDIR}/logs/download.log","r") as f:
            print("".join(f.readlines()[-200:]))
    except FileNotFoundError:
        print("Kein Logfile gefunden.")
    raise SystemExit(p.returncode)
print("✅ Download abgeschlossen.")


>> freqtrade download-data -c hard_user_data/configs/config_FundingCarrySpotProxyStrategy.json --userdir hard_user_data --datadir hard_user_data/data/bitstamp -t 15m 1h 5m --pairs BTC/USD --days 1000 --no-parallel-download -v --logfile hard_user_data/logs/download.log
2025-10-08 04:41:46,246 - freqtrade - INFO - freqtrade 2025.9.1
2025-10-08 04:41:47,977 - numexpr.utils - INFO - NumExpr defaulting to 2 threads.
2025-10-08 04:41:52,008 - freqtrade.configuration.load_config - INFO - Using config: hard_user_data/configs/config_FundingCarrySpotProxyStrategy.json ...
2025-10-08 04:41:52,012 - freqtrade.loggers - INFO - Enabling colorized output.
2025-10-08 04:41:52,013 - freqtrade.loggers - INFO - Logfile configured
2025-10-08 04:41:52,014 - freqtrade.loggers - INFO - Verbosity set to 1
2025-10-08 04:41:52,015 - freqtrade.configuration.configuration - INFO - Using user-data directory: hard_user_data ...
2025-10-08 04:41:52,016 - freqtrade.configuration.configuration - INFO - Using data dire

In [ ]:

# ✅ 13) Backtests, ZIP‑Mirror → Drive, Equity‑Kurven, Ranking
import os, glob, zipfile, json, pandas as pd, numpy as np, subprocess, shlex, shutil
from datetime import datetime

ALL_TRADES=[]; ALL_SUMMARY=[]

def parse_close_time(df):
    for c in ["close_time","close_date"]:
        if c in df.columns:
            dt = pd.to_datetime(df[c], errors="coerce", utc=True)
            if dt.notna().any(): return dt
    if "close_timestamp" in df.columns:
        ts = pd.to_numeric(df["close_timestamp"], errors="coerce")
        ts = np.where(ts > 10**12, ts/1000.0, ts)
        dt = pd.to_datetime(ts, unit="s", utc=True, errors="coerce")
        return dt
    return pd.Series([pd.NaT]*len(df), dtype="datetime64[ns, UTC]")

def build_equity_curve(trades_df, starting_balance):
    df = trades_df.copy()
    dt = parse_close_time(df)
    df = df.assign(_close_dt=dt).sort_values("_close_dt")
    pnl = pd.to_numeric(df.get("profit_abs", 0.0), errors="coerce").fillna(0.0)
    equity = starting_balance + pnl.cumsum()
    curve = pd.DataFrame({"timestamp": df["_close_dt"].values, "equity": equity.values}).dropna()
    curve = curve[curve["timestamp"].notna()]
    curve = curve.set_index("timestamp").sort_index()
    return curve

def daily_sharpe(curve, rf=0.0):
    if curve.empty: return 0.0, 0.0
    daily = curve["equity"].resample("1D").ffill()
    rets = daily.pct_change().dropna()
    if rets.empty: return 0.0, 0.0
    ex = rets - (rf/365.0)
    mu = ex.mean()
    sd = ex.std(ddof=1)
    sharpe = float((mu / sd) * np.sqrt(365)) if sd > 0 else float(mu * np.sqrt(365))
    max_dd = float((daily / daily.cummax() - 1.0).min())
    return sharpe, max_dd

def run_backtest(strat_class, cfgp, tf, tfd=None):
    export_dir = f"{USERDIR}/backtest_results/{EXCHANGE}_{PAIR.replace('/','_')}_{strat_class}_{tf}"
    os.makedirs(export_dir, exist_ok=True)
    logf = f"{USERDIR}/logs/backtest_{strat_class}_{tf}.log"
    cmd = [
        "freqtrade","backtesting",
        "-c", cfgp,
        "--userdir", USERDIR,
        "-s", strat_class,
        "-i", tf,
        "--datadir", DATADIR,
        "--export","trades",
        "--export-directory", export_dir,
        "--starting-balance", str(STARTING_BALANCE),
        "--fee", str(FEE),
        "-v","--logfile", logf
    ]
    if isinstance(tfd,str) and tfd.strip():
        cmd += ["--timeframe-detail", tfd.strip()]
    print(">>", " ".join(shlex.quote(c) for c in cmd))
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode != 0:
        print(f"⚠️ Backtest fehlgeschlagen: {strat_class} @ {tf}.")
        return None

    zips = sorted(glob.glob(os.path.join(export_dir,"*.zip")), key=os.path.getmtime, reverse=True)
    if not zips:
        return None
    z = zips[0]

    # ZIP nach Drive kopieren
    if GDRIVE_ENABLE and DRIVE_ZIPS:
        try:
            shutil.copy2(z, os.path.join(DRIVE_ZIPS, os.path.basename(z)))
            print("ZIP gespiegelt → Drive:", os.path.join(DRIVE_ZIPS, os.path.basename(z)))
        except Exception as e:
            print("ZIP‑Mirror Fehler:", e)

    # Trades & Summary laden
    with zipfile.ZipFile(z) as zz:
        members = zz.namelist()
        j = [m for m in members if m.endswith(".json")]
        tms = [m for m in j if "trades" in m]
        repm = [m for m in j if "backtest-result" in m or m.endswith("/backtest-result.json")]
        trades = []
        summ = {}
        if tms:
            with zz.open(tms[0]) as f:
                try: trades = json.load(f)
                except Exception: trades = []
        if repm:
            with zz.open(repm[0]) as f:
                rep = json.load(f)
            summ = rep.get("strategy", {}).get("summary", rep.get("summary", {})) or {}

    if trades:
        tdf = pd.DataFrame(trades)
        curve = build_equity_curve(tdf, STARTING_BALANCE)
        eq_path = f"{USERDIR}/equity_curves/{strat_class}_{tf}_equity.csv"
        curve.to_csv(eq_path)
        if GDRIVE_ENABLE and DRIVE_EQUITY:
            try:
                shutil.copy2(eq_path, os.path.join(DRIVE_EQUITY, os.path.basename(eq_path)))
            except Exception as e:
                print("Equity‑Mirror Fehler:", e)
        sharpe_tn, max_dd = daily_sharpe(curve, rf=RISK_FREE_RATE)
        profit_total = float(pd.to_numeric(tdf.get("profit_abs", 0.0), errors="coerce").fillna(0.0).sum())
        win_rate = float((pd.to_numeric(tdf.get("profit_abs", 0.0), errors="coerce").fillna(0.0) > 0).mean())
        tdf["strategy_class"]=strat_class; tdf["timeframe"]=tf
        ALL_TRADES.append(tdf)
    else:
        curve=None; sharpe_tn=0.0; max_dd=0.0
        profit_total = float(summ.get("profit_abs", 0.0)) if isinstance(summ.get("profit_abs", 0.0),(int,float)) else 0.0
        win_rate = float(summ.get("winrate", 0.0)) if isinstance(summ.get("winrate", 0.0),(int,float)) else 0.0

    row = {
        "strategy_class": strat_class,
        "timeframe": tf,
        "timeframe_detail": tfd or "",
        "trades": int(len(trades)) if trades else int(summ.get("trades", 0) or 0),
        "profit_total": profit_total,
        "win_rate": win_rate,
        "sharpe_tn": sharpe_tn,
        "max_dd": max_dd
    }
    ALL_SUMMARY.append(row)
    return row

# Läufe
for strat_class, meta in CONFIGS_BY_STRATEGY.items():
    cfgp = meta["config"]; tfd = meta.get("timeframe_detail")
    for tf in RUN_TIMEFRAMES:
        _ = run_backtest(strat_class, cfgp, tf, tfd=tfd)

# Aggregation & CSV Mirror
summary_df = pd.DataFrame(ALL_SUMMARY) if ALL_SUMMARY else pd.DataFrame()
if not summary_df.empty:
    metric = TOPN_METRIC
    if metric not in summary_df.columns: summary_df[metric]=np.nan
    filtered = summary_df[ summary_df["trades"].fillna(0) >= TOPN_MIN_TRADES ].copy()
    ranked = filtered.sort_values(by=[metric,"profit_total","max_dd"], ascending=[False, False, True])
    out_summary = f"{USERDIR}/backtest_results/_summary_multi_strategy_ranked.csv"
    ranked.to_csv(out_summary, index=False)
    print("📄 Summary CSV:", out_summary)
    if GDRIVE_ENABLE and DRIVE_RESULTS:
        try: shutil.copy2(out_summary, os.path.join(DRIVE_RESULTS, os.path.basename(out_summary)))
        except Exception as e: print("Summary‑Mirror Fehler:", e)
    display(ranked.head(25))
else:
    ranked = pd.DataFrame()
    print("Keine Summary-Daten.")

if ALL_TRADES:
    trades_all = pd.concat(ALL_TRADES, ignore_index=True, sort=False)
    out_trades = f"{USERDIR}/backtest_results/_all_trades_multi_strategy.csv"
    trades_all.to_csv(out_trades, index=False)
    print("📄 Trades CSV:", out_trades)
    if GDRIVE_ENABLE and DRIVE_RESULTS:
        try: shutil.copy2(out_trades, os.path.join(DRIVE_RESULTS, os.path.basename(out_trades)))
        except Exception as e: print("Trades‑Mirror Fehler:", e)
else:
    print("Keine Trades gesammelt.")


>> freqtrade backtesting -c hard_user_data/configs/config_FundingCarrySpotProxyStrategy.json --userdir hard_user_data -s FundingCarrySpotProxyStrategy_SpotOnly -i 5m --datadir hard_user_data/data/bitstamp --export trades --export-directory hard_user_data/backtest_results/bitstamp_BTC_USD_FundingCarrySpotProxyStrategy_SpotOnly_5m --starting-balance 10000.0 --fee 0.001 -v --logfile hard_user_data/logs/backtest_FundingCarrySpotProxyStrategy_SpotOnly_5m.log
2025-10-08 04:42:27,230 - freqtrade - INFO - freqtrade 2025.9.1
2025-10-08 04:42:27,516 - numexpr.utils - INFO - NumExpr defaulting to 2 threads.
2025-10-08 04:42:31,406 - freqtrade.configuration.load_config - INFO - Using config: hard_user_data/configs/config_FundingCarrySpotProxyStrategy.json ...
2025-10-08 04:42:31,409 - freqtrade.loggers - INFO - Enabling colorized output.
2025-10-08 04:42:31,410 - freqtrade.loggers - INFO - Logfile configured
2025-10-08 04:42:31,410 - freqtrade.loggers - INFO - Verbosity set to 1
2025-10-08 04:42:3

,strategy_class,timeframe,timeframe_detail,trades,profit_total,win_rate,sharpe_tn,max_dd


Keine Trades gesammelt.


In [ ]:

import json, glob, pathlib, shutil, os

STRATEGY_DIR = os.path.join(USERDIR, "strategies")
os.makedirs(STRATEGY_DIR, exist_ok=True)
CONFIG_DIR = USERDIR  # einfache Heuristik: configs liegen i. d. R. hier
HPO_CFG_DIR = os.path.join(USERDIR, "configs_hyperopt")
os.makedirs(HPO_CFG_DIR, exist_ok=True)

# Heuristik: Klassenname(n) aus .py-Dateien parsen
def _discover_strategy_classes(py_path: str):
    try:
        import ast
        with open(py_path, "r", encoding="utf-8") as f:
            tree = ast.parse(f.read(), filename=py_path)
        classes = [n.name for n in tree.body if isinstance(n, ast.ClassDef)]
        # Filter: übliche Namensmuster
        classes = [c for c in classes if c.lower().endswith("strategy") or "Strategy" in c]
        return classes
    except Exception as e:
        print("Parse-Fehler:", py_path, e)
        return []

# Configs nach Strategiezuordnung scannen
def _map_configs_to_strategies():
    mapping = {}  # {strategy_class: config_path}
    jsons = glob.glob(os.path.join(CONFIG_DIR, "config*.json")) + glob.glob(os.path.join(CONFIG_DIR, "*.json"))
    # Entferne offensichtliche Artefakte
    jsons = [p for p in jsons if "hyperopt" not in p and "validate" not in p and "pairs" not in p]
    for jp in jsons:
        try:
            with open(jp, "r", encoding="utf-8") as f:
                cfg = json.load(f)
        except Exception:
            continue
        # Einfache Fälle
        st = cfg.get("strategy")
        if st:
            mapping.setdefault(st, jp)
        # Strategie-Listen
        sl = cfg.get("strategy_list") or cfg.get("strategies")
        if isinstance(sl, list):
            for stn in sl:
                mapping.setdefault(stn, jp)
    return mapping

# 1) Strategien entdecken
STRATEGY_FILES = glob.glob(os.path.join(STRATEGY_DIR, "*.py"))
STRATEGY_CLASSES = []
for py in STRATEGY_FILES:
    STRATEGY_CLASSES += _discover_strategy_classes(py)
STRATEGY_CLASSES = sorted(set(STRATEGY_CLASSES))

# 2) Config-Zuordnung
CFG_BY_STRAT = _map_configs_to_strategies()

# 3) Hyperopt‑Schatten‑Config erzeugen (ohne Überschreiber)
HYPEROPT_LOSS = os.environ.get("HPO_LOSS","SharpeHyperOptLossDaily")
def make_hpo_config(src_cfg_path: str, tf_hint: str|None=None):
    with open(src_cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)
    # Sicherstellen: userdir/datadir gesetzt (werden via CLI überschrieben, schadet aber nicht)
    cfg.setdefault("user_data_dir", USERDIR)
    cfg.setdefault("dataformat_ohlcv", "json")

    # Hyperopt Loss eintragen
    cfg["hyperopt_loss"] = HYPEROPT_LOSS

    # Überschreiber entfernen – Hyperopt soll diese Felder optimieren dürfen
    for k in ["stoploss",
              "trailing_stop","trailing_stop_positive","trailing_stop_positive_offset",
              "trailing_only_offset_is_reached","trailing_stop_positive_offset_ratio"]:
        if k in cfg: cfg.pop(k, None)

    # timeframe ergänzen (falls fehlt)
    if tf_hint and "timeframe" not in cfg:
        cfg["timeframe"] = tf_hint

    outp = os.path.join(HPO_CFG_DIR, f"config_{pathlib.Path(src_cfg_path).stem}.hyperopt.json")
    with open(outp, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    return outp

print("Gefundene Strategien:", STRATEGY_CLASSES)
print("Config-Zuordnung:", CFG_BY_STRAT)


Gefundene Strategien: ['EventArbManualStrategy', 'EventArbManualStrategy_SpotOnly', 'FundingCarrySpotProxyStrategy', 'FundingCarrySpotProxyStrategy_SpotOnly', 'MacroRegimeTrendStrategy', 'MacroRegimeTrendStrategy_SpotOnly', 'MicroBreakout1mStrategy', 'MicroBreakout1mStrategy_SpotOnly', 'QualityMomentumLongOnlyStrategy', 'QualityMomentumLongOnlyStrategy_SpotOnly', 'SampleStrategy', 'ShortRealizedVolMeanRevertStrategy', 'ShortRealizedVolMeanRevertStrategy_SpotOnly', 'TSMOM12Strategy', 'TSMOM12Strategy_SpotOnly', 'ValueDiscount200Strategy', 'ValueDiscount200Strategy_SpotOnly']
Config-Zuordnung: {}


## ✅ 14) Hyperopt – Auto‑Run nach Backtest & Drive‑Mirror
Erzeugt Schatten‑Configs, führt Hyperopt (default + optional trailing) aus und spiegelt Parameter/Ergebnisse nach Drive.

In [ ]:

# ✅ 14a) Hyperopt‑Prep
import os, json, shutil, time, glob, subprocess, shlex, pathlib, re
from datetime import datetime

HYPEROPT_EPOCHS_DEFAULT   = int(os.environ.get("HPO_EPOCHS", "200"))
HYPEROPT_EARLY_STOP       = int(os.environ.get("HPO_EARLY_STOP", "60"))
HYPEROPT_MIN_TRADES       = int(os.environ.get("HPO_MIN_TRADES", "5"))
HYPEROPT_LOSS             = os.environ.get("HPO_LOSS", "SharpeHyperOptLossDaily")
HYPEROPT_SPACES           = os.environ.get("HPO_SPACES", "default")
HYPEROPT_RUN_TRAILING     = bool(int(os.environ.get("HPO_RUN_TRAILING", "1")))
HYPEROPT_TRAILING_EPOCHS  = int(os.environ.get("HPO_TRAILING_EPOCHS", "120"))
HYPEROPT_RANDOM_STATE     = int(os.environ.get("HPO_RANDOM_STATE", "0"))

HPO_LOCAL_RESULTS = os.path.join(USERDIR, "hyperopt_results")
HPO_LOCAL_PARAMS  = os.path.join(USERDIR, "hyperopt_params")
HPO_LOCAL_LOGS    = os.path.join(USERDIR, "logs")

DRIVE_HPO_RESULTS = os.path.join(GDRIVE_BASE, "hyperopt_results") if GDRIVE_ENABLE else None
DRIVE_HPO_PARAMS  = os.path.join(GDRIVE_BASE, "hyperopt_params") if GDRIVE_ENABLE else None
DRIVE_LOGS        = os.path.join(GDRIVE_BASE, "logs") if GDRIVE_ENABLE else None

for d in [HPO_LOCAL_RESULTS, HPO_LOCAL_PARAMS, HPO_LOCAL_LOGS]:
    os.makedirs(d, exist_ok=True)
if GDRIVE_ENABLE:
    for d in [DRIVE_HPO_RESULTS, DRIVE_HPO_PARAMS, DRIVE_LOGS]:
        if d: os.makedirs(d, exist_ok=True)

OVERRIDE_KEYS_TRAILING = [
    "trailing_stop", "trailing_stop_positive", "trailing_stop_positive_offset",
    "trailing_only_offset_is_reached", "trailing_stop_positive_offset_ratio", "trailing_only_offset_is_reached"
]
OVERRIDE_KEYS_STOPLOSS = ["stoploss"]
OVERRIDE_KEYS_TRADES   = ["max_open_trades"]

def make_hyperopt_config(src_cfg_path: str, spaces: str, ensure_loss: str, tf_hint: str|None=None):
    with open(src_cfg_path, "r", encoding="utf-8") as f:
        cfg = json.load(f)

    diff = {"removed": [], "set": {}}
    spaces_set = set(s.strip().lower() for s in spaces.split(","))

    def rm(key):
        if key in cfg:
            cfg.pop(key, None); diff["removed"].append(key)

    if "all" in spaces_set or "default" in spaces_set or "trailing" in spaces_set:
        for k in OVERRIDE_KEYS_TRAILING: rm(k)
    if "all" in spaces_set or "default" in spaces_set or "stoploss" in spaces_set:
        for k in OVERRIDE_KEYS_STOPLOSS: rm(k)
    if "all" in spaces_set or "trades" in spaces_set:
        for k in OVERRIDE_KEYS_TRADES: rm(k)

    cfg["hyperopt_loss"] = ensure_loss
    diff["set"]["hyperopt_loss"] = ensure_loss

    if tf_hint and "timeframe" not in cfg:
        cfg["timeframe"] = tf_hint
        diff["set"]["timeframe"] = tf_hint

    base = pathlib.Path(src_cfg_path).name
    out_name = base.replace(".json", f".hyperopt.{spaces}.json")
    out_dir = os.path.join(USERDIR, "configs_hyperopt")
    os.makedirs(out_dir, exist_ok=True)
    out_path = os.path.join(out_dir, out_name)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)
    return out_path, diff

def latest_file_in(dirpath, pattern="*"):
    files = glob.glob(os.path.join(dirpath, pattern))
    return max(files, key=os.path.getmtime) if files else None

def mirror_to_drive(local_path: str, drive_dir: str|None):
    if not (GDRIVE_ENABLE and drive_dir and os.path.exists(local_path)):
        return
    try:
        shutil.copy2(local_path, os.path.join(drive_dir, os.path.basename(local_path)))
    except Exception as e:
        print("Drive‑Mirror Fehler:", e)

def run_hyperopt_for(strat_class: str, cfg_path: str, timeframe: str, spaces: str, epochs: int):
    os.makedirs(HPO_LOCAL_RESULTS, exist_ok=True)
    start_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    log_file = os.path.join(HPO_LOCAL_LOGS, f"hyperopt_{strat_class}_{timeframe}_{spaces}_{start_ts}.log")
    results_before = set(glob.glob(os.path.join(USERDIR, "hyperopt_results", "*.json")))
    params_before  = set(glob.glob(os.path.join(USERDIR, "strategies", "*.json")))

    cmd = [
        "freqtrade","hyperopt",
        "-c", cfg_path,
        "--userdir", USERDIR,
        "--datadir", DATADIR,
        "--strategy", strat_class,
        "-i", timeframe,
        "-e", str(epochs),
        "--spaces", spaces,
        "--hyperopt-loss", HYPEROPT_LOSS,
        "--min-trades", str(HYPEROPT_MIN_TRADES),
        "--ignore-missing-spaces",
        "--print-json",
        "-j","-1"
    ]
    if HYPEROPT_EARLY_STOP > 0:
        cmd += ["--early-stop", str(HYPEROPT_EARLY_STOP)]
    if HYPEROPT_RANDOM_STATE > 0:
        cmd += ["--random-state", str(HYPEROPT_RANDOM_STATE)]

    print(f"[Hyperopt] {strat_class} @ {timeframe} spaces={spaces} epochs={epochs}")
    print(">>", " ".join(shlex.quote(c) for c in cmd))

    with open(log_file, "w", encoding="utf-8") as lf:
        subprocess.run(cmd, stdout=lf, stderr=subprocess.STDOUT, text=True)

    res_after = set(glob.glob(os.path.join(USERDIR, "hyperopt_results", "*.json")))
    new_res_files = sorted(list(res_after - results_before), key=os.path.getmtime)
    latest_res = new_res_files[-1] if new_res_files else latest_file_in(os.path.join(USERDIR, "hyperopt_results"), "*.json")

    params_after = set(glob.glob(os.path.join(USERDIR, "strategies", "*.json")))
    new_param_files = sorted(list(params_after - params_before), key=os.path.getmtime)
    latest_param = new_param_files[-1] if new_param_files else None

    mirror_to_drive(log_file, DRIVE_LOGS)
    if latest_res: mirror_to_drive(latest_res, DRIVE_HPO_RESULTS)
    if latest_param: mirror_to_drive(latest_param, DRIVE_HPO_PARAMS)

    out = {"log": log_file, "hyperopt_result_file": latest_res, "param_json": latest_param}
    if latest_param:
        out["param_json_name"] = os.path.basename(latest_param)
    return out


In [ ]:

# ✅ 14b) Hyperopt‑Runs über alle Strategien (post‑Backtest)
import pandas as pd, json, os

HPO_SUMMARY = []

for strat_class, meta in CONFIGS_BY_STRATEGY.items():
    base_cfg = meta["config"]
    for tf in RUN_TIMEFRAMES:
        cfg_hp, diff = make_hyperopt_config(base_cfg, HYPEROPT_SPACES, HYPEROPT_LOSS, tf_hint=tf)
        print(f"\n[{strat_class} @ {tf}] Schatten‑Config:", cfg_hp, "| diff:", diff)

        res1 = run_hyperopt_for(strat_class, cfg_hp, tf, HYPEROPT_SPACES, HYPEROPT_EPOCHS_DEFAULT)
        HPO_SUMMARY.append({
            "strategy": strat_class, "timeframe": tf, "spaces": HYPEROPT_SPACES,
            "param_json": res1.get("param_json"), "result_file": res1.get("hyperopt_result_file"),
            "log": res1.get("log")
        })

        if HYPEROPT_RUN_TRAILING:
            cfg_tr, diff_tr = make_hyperopt_config(base_cfg, "trailing", HYPEROPT_LOSS, tf_hint=tf)
            print(f"[{strat_class} @ {tf}] Schatten‑Config (trailing):", cfg_tr, "| diff:", diff_tr)
            res2 = run_hyperopt_for(strat_class, cfg_tr, tf, "trailing", HYPEROPT_TRAILING_EPOCHS)
            HPO_SUMMARY.append({
                "strategy": strat_class, "timeframe": tf, "spaces": "trailing",
                "param_json": res2.get("param_json"), "result_file": res2.get("hyperopt_result_file"),
                "log": res2.get("log")
            })

hpo_df = pd.DataFrame(HPO_SUMMARY)
out_csv = os.path.join(USERDIR, "hyperopt_results", "_hyperopt_runs.csv")
hpo_df.to_csv(out_csv, index=False)
print("\n📄 Hyperopt‑Runs CSV:", out_csv)

if GDRIVE_ENABLE and DRIVE_HPO_RESULTS:
    try:
        import shutil
        shutil.copy2(out_csv, os.path.join(DRIVE_HPO_RESULTS, os.path.basename(out_csv)))
    except Exception as e:
        print("Drive‑Mirror Fehler (CSV):", e)



[FundingCarrySpotProxyStrategy_SpotOnly @ 5m] Schatten‑Config: hard_user_data/configs_hyperopt/config_FundingCarrySpotProxyStrategy.hyperopt.default.json | diff: {'removed': [], 'set': {'hyperopt_loss': 'SharpeHyperOptLossDaily'}}
[Hyperopt] FundingCarrySpotProxyStrategy_SpotOnly @ 5m spaces=default epochs=200
>> freqtrade hyperopt -c hard_user_data/configs_hyperopt/config_FundingCarrySpotProxyStrategy.hyperopt.default.json --userdir hard_user_data --datadir hard_user_data/data/bitstamp --strategy FundingCarrySpotProxyStrategy_SpotOnly -i 5m -e 200 --spaces default --hyperopt-loss SharpeHyperOptLossDaily --min-trades 5 --ignore-missing-spaces --print-json -j -1 --early-stop 60


/tmp/ipython-input-1065475825.py:83: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


[FundingCarrySpotProxyStrategy_SpotOnly @ 5m] Schatten‑Config (trailing): hard_user_data/configs_hyperopt/config_FundingCarrySpotProxyStrategy.hyperopt.trailing.json | diff: {'removed': [], 'set': {'hyperopt_loss': 'SharpeHyperOptLossDaily'}}
[Hyperopt] FundingCarrySpotProxyStrategy_SpotOnly @ 5m spaces=trailing epochs=120
>> freqtrade hyperopt -c hard_user_data/configs_hyperopt/config_FundingCarrySpotProxyStrategy.hyperopt.trailing.json --userdir hard_user_data --datadir hard_user_data/data/bitstamp --strategy FundingCarrySpotProxyStrategy_SpotOnly -i 5m -e 120 --spaces trailing --hyperopt-loss SharpeHyperOptLossDaily --min-trades 5 --ignore-missing-spaces --print-json -j -1 --early-stop 60


/tmp/ipython-input-1065475825.py:83: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")



[FundingCarrySpotProxyStrategy_SpotOnly @ 15m] Schatten‑Config: hard_user_data/configs_hyperopt/config_FundingCarrySpotProxyStrategy.hyperopt.default.json | diff: {'removed': [], 'set': {'hyperopt_loss': 'SharpeHyperOptLossDaily'}}
[Hyperopt] FundingCarrySpotProxyStrategy_SpotOnly @ 15m spaces=default epochs=200
>> freqtrade hyperopt -c hard_user_data/configs_hyperopt/config_FundingCarrySpotProxyStrategy.hyperopt.default.json --userdir hard_user_data --datadir hard_user_data/data/bitstamp --strategy FundingCarrySpotProxyStrategy_SpotOnly -i 15m -e 200 --spaces default --hyperopt-loss SharpeHyperOptLossDaily --min-trades 5 --ignore-missing-spaces --print-json -j -1 --early-stop 60


...

/tmp/ipython-input-1065475825.py:83: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")



[ShortRealizedVolMeanRevertStrategy_SpotOnly @ 5m] Schatten‑Config: hard_user_data/configs_hyperopt/config_ShortRealizedVolMeanRevertStrategy.hyperopt.default.json | diff: {'removed': [], 'set': {'hyperopt_loss': 'SharpeHyperOptLossDaily'}}
[Hyperopt] ShortRealizedVolMeanRevertStrategy_SpotOnly @ 5m spaces=default epochs=200
>> freqtrade hyperopt -c hard_user_data/configs_hyperopt/config_ShortRealizedVolMeanRevertStrategy.hyperopt.default.json --userdir hard_user_data --datadir hard_user_data/data/bitstamp --strategy ShortRealizedVolMeanRevertStrategy_SpotOnly -i 5m -e 200 --spaces default --hyperopt-loss SharpeHyperOptLossDaily --min-trades 5 --ignore-missing-spaces --print-json -j -1 --early-stop 60


/tmp/ipython-input-1065475825.py:83: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


[ShortRealizedVolMeanRevertStrategy_SpotOnly @ 5m] Schatten‑Config (trailing): hard_user_data/configs_hyperopt/config_ShortRealizedVolMeanRevertStrategy.hyperopt.trailing.json | diff: {'removed': [], 'set': {'hyperopt_loss': 'SharpeHyperOptLossDaily'}}
[Hyperopt] ShortRealizedVolMeanRevertStrategy_SpotOnly @ 5m spaces=trailing epochs=120
>> freqtrade hyperopt -c hard_user_data/configs_hyperopt/config_ShortRealizedVolMeanRevertStrategy.hyperopt.trailing.json --userdir hard_user_data --datadir hard_user_data/data/bitstamp --strategy ShortRealizedVolMeanRevertStrategy_SpotOnly -i 5m -e 120 --spaces trailing --hyperopt-loss SharpeHyperOptLossDaily --min-trades 5 --ignore-missing-spaces --print-json -j -1 --early-stop 60


/tmp/ipython-input-1065475825.py:83: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  start_ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
